In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import Imputer
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName('Olist Application Performance Optimization')\
.config('spark.executor.memory', '6g')\
.config('spark.executor.cores', '4')\
.config('spark.executor.instances', '2')\
.config('spark.driver.memory', '4g')\
.config('spark.driver.maxResultSize', '2g')\
.config('spark.sql.shuffle.partitons', '64')\
.config('spark.default.parallelism', '64')\
.config('spark.sql.adaptive.enabled', 'true')\
.config('spark.sql.adaptive.coalescePartition.enabled', 'true')\
.config('spark.sql.autoBroadcastJoinThreshold', 20*1024*1024)\
.config('spark.sql.files.maxPartitionBytes', '64MB')\
.config('spark.sql.files.openCostInBytes', '2MB')\
.config('spark.memory.fraction', 0.8)\
.config('spark.memory.storageFraction', 0.2)\
.enableHive/
.getOrCreate()

26/02/18 08:26:23 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
hdfs_path = '/data/olist/'

In [4]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv', header=True, inferSchema = True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv', header=True, inferSchema = True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv', header=True, inferSchema = True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv', header=True, inferSchema = True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv', header=True, inferSchema = True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv', header=True, inferSchema = True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema = True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv', header=True, inferSchema = True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv', header=True, inferSchema = True)

In [5]:
!hdfs dfs -ls -h /data/olist/processed

Found 11 items
-rw-r--r--   2 root hadoop          0 2026-02-18 08:25 /data/olist/processed/_SUCCESS
-rw-r--r--   2 root hadoop     31.6 M 2026-02-18 08:25 /data/olist/processed/part-00000-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.2 M 2026-02-18 08:25 /data/olist/processed/part-00001-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.0 M 2026-02-18 08:25 /data/olist/processed/part-00002-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     30.9 M 2026-02-18 08:25 /data/olist/processed/part-00003-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     31.4 M 2026-02-18 08:25 /data/olist/processed/part-00004-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop     18.2 M 2026-02-18 08:25 /data/olist/processed/part-00005-16b1628c-87f6-4c10-bedd-95f804aadf47-c000.snappy.parquet
-rw-r--r--   2 root hadoop   

In [6]:
full_orders_df = spark.read.parquet('/data/olist/processed')

In [7]:
full_orders_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

## Optimized Join Strategies

In [8]:
## Broadcast

customers_broadcast_df = broadcast(customers_df)
optimized_broadcast_join = full_orders_df.join(customers_broadcast_df, 'customer_id')

In [9]:
## Sort and Merge Join

sorted_customers_df = customers_df.sortWithinPartitions('customer_id')
sorted_orders_df = full_orders_df.sortWithinPartitions('customer_id')

optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customers_df, 'customer_id')

In [10]:
## Bucket Join --> used if we have some repeated queries on the large datasets

bucketed_customers_df = customers_df.repartition(10, 'customer_id')
bucketed_orders_df = full_orders_df.repartition(10, 'customer_id')

bucket_join_df = bucketed_orders_df.join(bucketed_customers_df, 'customer_id')

In [11]:
## Skew Join handling

skew_join_handled = full_orders_df.join(customers_df.hint('skew'), 'customer_id')

26/02/18 08:27:19 WARN HintErrorLogger: Unrecognized hint: skew()


## Data Serving

In [12]:
## Save as parquet in HDFS

full_orders_df.write.mode('overwrite').parquet('/data/olist/proc')

26/02/18 08:27:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [13]:
## Save in as a parquet in google cloud storage --> to see this data search google storage --> go to the view bucket list --> 
# --> go inside the bucket where we have saved the file --> there we can see the file.

full_orders_df.write.mode('overwrite').parquet("gs://dataproc-staging-us-central1-94003790817-ihavau21/temp_data")

In [ ]:
## Save dataframe as SQL table in Hive

full_orders_df.write.mode('overwrite').saveAsTable('full_orders_details')

In [ ]:
spark.sql('show tables')

In [16]:
## Save the dataframe as CSV

full_orders_df.write.mode('overwrite').option('header', 'true').csv('/data/olist/proc')

In [17]:
spark.stop()